In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from noisynetworks import IndependentNoisyLayer
from torchrl.modules import NoisyLinear

from collections import deque

import gymnasium as gym

import random

import matplotlib.pyplot as plt

In [2]:
from IPython import display
plt.ion()

In [3]:
neurons = 128

class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = NoisyLinear(n_observations, neurons)
        self.layer2 = NoisyLinear(neurons, n_actions)

    def forward(self, x):
        x = F.relu(input=self.layer1(x))
        return self.layer2(x)

In [4]:
device = "cuda"

env = gym.make("CartPole-v1", render_mode="rgb_array")
state, info = env.reset()

n_observations = len(state)
n_actions = env.action_space.n

In [5]:
current_model = DQN(n_observations=n_observations, n_actions=n_actions).to(device)
target_model = DQN(n_observations=n_observations, n_actions=n_actions).to(device)
target_model.load_state_dict(state_dict=current_model.state_dict())

learning_rate = 0.001
optimizer = torch.optim.AdamW(current_model.parameters(), lr=learning_rate, amsgrad=True)

In [6]:
memory_length = 10000

memory = deque([], maxlen=memory_length)

In [7]:
episode_durations = []

def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    if not show_result:
        display.display(plt.gcf())
        display.clear_output(wait=True)
    else:
        display.display(plt.gcf())

In [ ]:
N_episodes = 1000
batch_size = 128
target_updating_steps = 10
discount_factor = 0.99
TAU = 0.005

for episode in range(N_episodes):
    state, info = env.reset()
    state = torch.tensor(state).to(device)

    done = False

    t = 0

    while not done:
        with torch.no_grad():
            q_values = current_model(torch.tensor(state).to(device))
            action = torch.argmax(q_values)

        next_state, reward, terminated, truncated, _ = env.step(action.item())
        
        reward = torch.tensor([reward]).to(device)
        
        done = terminated or truncated
        
        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
        memory.append([state, action, reward, next_state])
        state = next_state

        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states = zip(*batch)

            q_values = current_model(states).gather(1, actions)

            with torch.no_grad():
                next_q_values = target_model(next_states).max(dim=1)[0]
            targets = rewards + discount_factor * next_q_values

            criterion = nn.SmoothL1Loss()
            loss = criterion(q_values, targets.unsqueeze(1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_value_(current_model.parameters(), 100)
            optimizer.step()

            target_model_state_dict = target_model.state_dict()
            current_model_state_dict = current_model.state_dict()
            for key in current_model_state_dict:
                target_model_state_dict[key] = current_model_state_dict[key]*TAU + target_model_state_dict[key]*(1-TAU)
            target_model.load_state_dict(target_model_state_dict)

            t += 1

            if done:
                episode_durations.append(t + 1)
                plot_durations()
                break


print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()

C:\Users\natha\AppData\Local\Temp\ipykernel_23896\3187186607.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q_values = current_model(torch.tensor(state).to(device))


(tensor([[0.0064, 0.1705, 0.1411, 0.2140]], device='cuda:0'), tensor([[ 0.2725,  2.1360,  0.0373, -1.9446]], device='cuda:0'), tensor([[-0.0027,  0.2292,  0.1098,  0.0072]], device='cuda:0'), tensor([[0.0033, 0.1741, 0.1292, 0.1319]], device='cuda:0'), tensor([[ 0.0337, -0.3943,  0.0321,  0.6353]], device='cuda:0'), tensor([[ 0.0206, -0.1482,  0.0240,  0.3092]], device='cuda:0'), tensor([[0.0071, 0.0295, 0.1181, 0.4034]], device='cuda:0'), tensor([[0.0346, 0.3916, 0.1644, 0.2215]], device='cuda:0'), tensor([[ 0.0258, -0.1996,  0.0448,  0.3529]], device='cuda:0'), tensor([[0.0204, 0.3592, 0.1508, 0.0599]], device='cuda:0'), tensor([[-2.2285e-04, -1.5798e-01,  9.3954e-02,  5.2684e-01]], device='cuda:0'), tensor([[-0.0098, -0.1652,  0.0912,  0.4678]], device='cuda:0'), tensor([[-0.0105,  0.2155,  0.1278,  0.0917]], device='cuda:0'), tensor([[0.0019, 0.0327, 0.1099, 0.3324]], device='cuda:0'), tensor([[0.0171, 0.1664, 0.1447, 0.3037]], device='cuda:0'), tensor([[ 0.0176, -0.3437,  0.0302, 

TypeError: linear(): argument 'input' (position 1) must be Tensor, not tuple